# EDM Classifier — Colab launcher

This notebook is only a **launcher**: it clones the repo, installs the
`classifier` package, mounts Google Drive and calls the same package that runs
locally. No project logic lives here.

**Runtime → Change runtime type → GPU** before running.

## 1. Clone the repo and install the package

In [ ]:
import os
if not os.path.isdir('edm-classifier'):
    !git clone https://github.com/Nestiii/edm-classifier.git
else:
    !cd edm-classifier && git pull
# Non-editable install: importable immediately, no kernel restart needed in Colab.
%pip install ./edm-classifier/classifier

## 2. Mount Google Drive (where the raw dataset lives)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Point these at your Drive layout: RAW = one folder per subgenre.
RAW_DIR   = '/content/drive/MyDrive/edm-dataset/raw'
WORK_DIR  = '/content/drive/MyDrive/edm-dataset/work'
MANIFEST  = f'{WORK_DIR}/manifest.csv'
SPLITS    = f'{WORK_DIR}/splits.json'
CACHE_DIR = f'{WORK_DIR}/cache'

## 3. Build the manifest and validate the dataset (WBS 2.4 / 2.3)

In [ ]:
from edm_classifier.data.manifest import build_manifest, save_manifest
from edm_classifier.data.validate import validate_manifest

entries = build_manifest(RAW_DIR)
save_manifest(entries, MANIFEST)
print(validate_manifest(entries).summary())

## 4. Persist the split and precompute the feature cache (WBS 4.3)

In [ ]:
from edm_classifier.data.manifest import load_manifest, to_track_records
from edm_classifier.data.splits import stratified_split, save_split
from edm_classifier.data.preprocess import preprocess_dataset

records = to_track_records(load_manifest(MANIFEST))
split = stratified_split(records)
save_split(split, SPLITS)
print('split sizes:', split.sizes())

result = preprocess_dataset(records, CACHE_DIR,
                            progress=lambda d, t: print(f'{d}/{t}') if d % 25 == 0 else None)
print(result)

## 5. Train on the GPU (WBS 4.4)

In [ ]:
from edm_classifier.training.train import TrainConfig, train_model
import json

RUN_DIR = f'{WORK_DIR}/run'
config = TrainConfig(epochs=50, batch_size=32, learning_rate=1e-3, n_channels=128)
report = train_model(CACHE_DIR, SPLITS, RUN_DIR, config=config, device='auto')
print(json.dumps(report, indent=2))
# Best checkpoint -> {RUN_DIR}/model.pt ; metrics -> {RUN_DIR}/report.json